# Conversation Buffer Memory

> **The most basic form of agent memory: store the entire conversation word-for-word.**

Every AI agent needs memory. Without it, each turn is a blank slate. The agent can't reference what you said a minute ago, let alone hold a coherent multi-turn conversation.

Think of a tape recorder that never pauses. It captures every word, and when the agent needs to reply, it replays the full tape. That's **Conversation Buffer Memory**. It's the foundation that every other memory technique builds on. The idea is straightforward:

1. Keep an ordered list of messages.
2. Send the *entire* list to the LLM on every turn.
3. Append the response. Repeat.

This notebook shows you how to build it from scratch with the **Anthropic SDK** (Anthropic's Python library for calling Claude). You'll visualize the linear token-growth problem with **matplotlib**. You'll also compare the DIY version with **LangChain's built-in ConversationBufferMemory**.

**By the end you'll understand:**
- Why buffer memory is the default starting point for any agent.
- How messages accumulate and why cost and latency grow linearly per turn.
- When this approach breaks down and what to do next.

## Key Concepts

- **Message list**: An ordered list of `{role, content}` dicts. Roles are typically `user` and `assistant` (plus an optional `system` prompt).
- **Role alternation**: The LLM relies on correct role sequencing to know who said what.
- **Context injection**: The full message list is passed as the prompt on every API call. "Context" means the text the model reads before answering.
- **Token counting**: Each message consumes tokens (word-pieces the model uses internally). The running total determines cost and whether you'll hit the context-window ceiling.
- **Linear growth**: Every new turn adds tokens. *All* previous tokens are re-sent too. So cumulative token usage grows **O(n²)** across turns, where *n* is the number of turns. This means that doubling the turns roughly quadruples the total cost.

## Architecture

<p align="center">
 <img src="../../images/diagrams/01_conversation_buffer_memory.svg" alt="diagram" width="720"/>
</p>

<details><summary>Mermaid source</summary>

```mermaid
sequenceDiagram
 participant U as User
 participant B as Buffer Memory
 participant L as LLM (Claude)

 U->>B: "Hi, I'm Alice"
 B->>B: Append {role: user, content: "Hi, I'm Alice"}
 B->>L: [msg1]
 L-->>B: "Hello Alice! How can I help?"
 B->>B: Append {role: assistant, content: "Hello Alice!..."}

 U->>B: "What's my name?"
 B->>B: Append {role: user, content: "What's my name?"}
 B->>L: [msg1, msg2, msg3]
 L-->>B: "Your name is Alice."
 B->>B: Append {role: assistant, content: "Your name is Alice."}

 Note over B: Buffer grows with every turn.<br/>All messages re-sent each call.
```

</details>

## Setup

Install dependencies and configure API access.

In [ ]:
%pip install -q anthropic python-dotenv matplotlib langchain langchain-anthropic

Import the Anthropic SDK and standard library helpers. The API key loads from a `.env` file.

In [ ]:
import os
import json
import copy
from dotenv import load_dotenv

load_dotenv() # reads ANTHROPIC_API_KEY from .env

import anthropic

assert os.getenv("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in your .env file"

## Implementation

We'll build a minimal `ConversationBufferMemory` class that:
1. Stores messages in a plain Python list.
2. Exposes `add_user_message` / `add_assistant_message` helpers.
3. Sends the full history to the Anthropic API on each call.
4. Tracks token usage per turn for visualization later.

In [ ]:
class ConversationBufferMemory:
 """Minimal conversation buffer memory built on the Anthropic SDK."""

 def __init__(
 self,
 model: str = "claude-sonnet-4-20250514",
 system_prompt: str | None = None,
 max_tokens: int = 1024,
 ):
 self.client = anthropic.Anthropic()
 self.model = model
 self.system_prompt = system_prompt
 self.max_tokens = max_tokens

 # Core data structure: the message buffer
 self.messages: list[dict] = []

 # Bookkeeping for the token-growth demo
 self.turn_token_usage: list[dict] = []

 # ── Message helpers ──────────────────────────────────────────────
 def add_user_message(self, content: str) -> None:
 self.messages.append({"role": "user", "content": content})

 def add_assistant_message(self, content: str) -> None:
 self.messages.append({"role": "assistant", "content": content})



Now we add the `chat` method and utility helpers. The `chat` method is where buffer memory does its work: it appends the user message, sends the **entire** message list to the API, and records token usage. Notice that `messages` (the full list) is passed every time. That's the defining trait of buffer memory.

In [ ]:
 # ── Chat ─────────────────────────────────────────────────────────
 def chat(self, user_input: str) -> str:
 """Send a message and get a response. The *entire* buffer is sent."""
 self.add_user_message(user_input)

 kwargs = dict(
 model=self.model,
 max_tokens=self.max_tokens,
 messages=self.messages, # <-- full history every time
 )
 if self.system_prompt:
 kwargs["system"] = self.system_prompt

 response = self.client.messages.create(**kwargs)

 assistant_text = response.content[0].text
 self.add_assistant_message(assistant_text)

 # Record token usage for this turn
 self.turn_token_usage.append({
 "turn": len(self.turn_token_usage) + 1,
 "input_tokens": response.usage.input_tokens,
 "output_tokens": response.usage.output_tokens,
 })

 return assistant_text

 # ── Utilities ────────────────────────────────────────────────────
 def get_history(self) -> list[dict]:
 return copy.deepcopy(self.messages)

 def clear(self) -> None:
 self.messages.clear()
 self.turn_token_usage.clear()

 def __len__(self) -> int:
 return len(self.messages)

 def __repr__(self) -> str:
 return f"ConversationBufferMemory({len(self.messages)} messages)"

## Example Run

A short multi-turn conversation shows buffer memory in action. The agent remembers everything because it re-reads the whole history every turn.

In [ ]:
memory = ConversationBufferMemory(
 system_prompt="You are a helpful, concise assistant. Keep replies under 2 sentences.",
)

# Multi-turn conversation
exchanges = [
 "Hi! My name is Alice and I'm a machine-learning engineer.",
 "I'm working on a project about agent memory. Any tips?",
 "What's my name and what do I do?", # recall test
]

for msg in exchanges:
 print(f"👤 User: {msg}")
 reply = memory.chat(msg)
 print(f"🤖 Agent: {reply}\n")

print(f"Buffer now contains {len(memory)} messages.")

Inspect the raw message buffer. This is exactly what the LLM receives on each call.

In [ ]:
# Peek at the raw message buffer
for i, msg in enumerate(memory.get_history()):
 role_label = "USER" if msg["role"] == "user" else "ASST"
 # Truncate long messages for display
 preview = msg["content"][:80] + ("..." if len(msg["content"]) > 80 else "")
 print(f" [{i}] {role_label}: {preview}")

## The Linear Token-Growth Problem

With buffer memory, **every previous message is re-sent on every turn**. Here's what that looks like:

| Turn | Tokens sent to the LLM |
|------|----------------------|
| 1 | the first message only |
| 2 | messages 1-3 |
| 3 | messages 1-5 |
| *n* | messages 1-(2n-1) |

Input tokens per turn grow **linearly** with conversation length. The **cumulative** tokens used across the whole conversation grow **quadratically**. Let's prove it with a longer conversation and a chart.

In [ ]:
# Run a longer conversation to see the growth curve
memory_demo = ConversationBufferMemory(
 system_prompt="You are a helpful assistant. Keep answers to 1-2 sentences.",
)

demo_messages = [
 "Hello! I'm Bob.",
 "I live in San Francisco.",
 "I have a golden retriever named Max.",
 "My favorite programming language is Python.",
 "I work at a startup that builds AI tools.",
 "We're building a chatbot with memory capabilities.",
 "Our team has 12 engineers.",
 "We use Claude as our primary LLM.",
 "What do you know about me so far?",
 "Summarize everything you remember about me in a list.",
]

for msg in demo_messages:
 reply = memory_demo.chat(msg)

print(f"Completed {len(memory_demo.turn_token_usage)} turns.")
print("\nToken usage per turn:")
for t in memory_demo.turn_token_usage:
 print(f" Turn {t['turn']:2d}: {t['input_tokens']:5d} input tokens, {t['output_tokens']:4d} output tokens")

Plot input tokens per turn and cumulative usage. You'll see the linear and quadratic growth patterns clearly.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

turns = [t["turn"] for t in memory_demo.turn_token_usage]
input_tokens = [t["input_tokens"] for t in memory_demo.turn_token_usage]
cumulative = np.cumsum(input_tokens)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Left: input tokens per turn (linear growth)
ax1.bar(turns, input_tokens, color="#6366f1", alpha=0.85)
ax1.set_xlabel("Turn")
ax1.set_ylabel("Input Tokens")
ax1.set_title("Input Tokens per Turn (Linear Growth)")
ax1.set_xticks(turns)

# Right: cumulative input tokens (quadratic growth)
ax2.plot(turns, cumulative, "o-", color="#f43f5e", linewidth=2, markersize=6)
ax2.fill_between(turns, cumulative, alpha=0.15, color="#f43f5e")
ax2.set_xlabel("Turn")
ax2.set_ylabel("Cumulative Input Tokens")
ax2.set_title("Cumulative Token Usage (Quadratic Growth)")
ax2.set_xticks(turns)

plt.tight_layout()
plt.savefig("token_growth.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nTotal input tokens across all turns: {int(cumulative[-1]):,}")
print(f"If using a window/summary approach, this could be drastically reduced.")

## Persistence

A production buffer memory must survive process restarts. The data structure is a list of dicts, so JSON serialization (converting it to a text file format) is straightforward.

In [ ]:
def save_conversation(memory: ConversationBufferMemory, path: str) -> None:
 """Persist the message buffer to a JSON file."""
 with open(path, "w") as f:
 json.dump(memory.get_history(), f, indent=2)
 print(f"Saved {len(memory)} messages → {path}")


def load_conversation(path: str, **kwargs) -> ConversationBufferMemory:
 """Restore a ConversationBufferMemory from a JSON file."""
 with open(path) as f:
 messages = json.load(f)
 mem = ConversationBufferMemory(**kwargs)
 mem.messages = messages
 print(f"Loaded {len(messages)} messages ← {path}")
 return mem


# Demo round-trip
save_conversation(memory, "conversation.json")
loaded = load_conversation(
 "conversation.json",
 system_prompt="You are a helpful, concise assistant. Keep replies under 2 sentences.",
)

# Verify the loaded memory still works
reply = loaded.chat("Remind me - what project was I working on?")
print(f"\n🤖 Agent (from loaded memory): {reply}")

## LangChain Comparison

LangChain provides a built-in `ConversationBufferMemory` that does the same thing. Let's see how the framework version compares to our scratch implementation.

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain.memory import ConversationBufferMemory as LCBufferMemory
from langchain.chains import ConversationChain

llm = ChatAnthropic(model="claude-sonnet-4-20250514", max_tokens=1024)

lc_memory = LCBufferMemory(return_messages=True)

chain = ConversationChain(
 llm=llm,
 memory=lc_memory,
 verbose=False,
)

# Same conversation
print("👤:", "Hi, my name is Charlie.")
print("🤖:", chain.predict(input="Hi, my name is Charlie."), "\n")

print("👤:", "I'm building a knowledge graph for my company.")
print("🤖:", chain.predict(input="I'm building a knowledge graph for my company."), "\n")

print("👤:", "What's my name and what am I building?")
print("🤖:", chain.predict(input="What's my name and what am I building?"), "\n")

# Inspect LangChain's internal buffer
print("--- LangChain buffer contents ---")
for msg in lc_memory.chat_memory.messages:
 role = msg.__class__.__name__.replace("Message", "")
 preview = msg.content[:80] + ("..." if len(msg.content) > 80 else "")
 print(f" {role}: {preview}")

**DIY vs. LangChain: side-by-side:**

| Aspect | Our Implementation | LangChain `ConversationBufferMemory` |
|--------|-------------------|--------------------------------------|
| **Data structure** | `list[dict]` with `role`/`content` | `ChatMessageHistory` with typed message objects |
| **API integration** | Direct `anthropic.Anthropic()` calls | Wrapped in `ConversationChain` |
| **Token tracking** | Manual via `response.usage` | Available through callbacks |
| **Persistence** | DIY JSON save/load | Built-in serialization options |
| **Flexibility** | Full control over prompt assembly | Opinionated chain structure |
| **Lines of code** | ~50 lines | ~5 lines (the framework does the rest) |

**Takeaway:** LangChain is convenient for prototyping. Building from scratch gives you full control over token budgets, multi-model routing, and non-standard message formats.

## Tradeoffs

### When Conversation Buffer Memory Works Well
- **Short conversations** (under 20 turns) where total tokens stay comfortably within the context window.
- **Prototyping**: get something working in minutes before optimizing.
- **Perfect recall required**: no information is ever dropped or summarized.

### When It Breaks Down
- **Long conversations**: input tokens per turn grow linearly. Cumulative cost grows quadratically.
- **Cost-sensitive applications**: re-sending the entire history on every call is expensive.
- **Latency-sensitive applications**: more input tokens means slower responses. Latency is the delay between your request and the model's reply.
- **Context window limits**: eventually the buffer exceeds the model's maximum (for example, 200k tokens for Claude).

### Cost Example
Consider a 50-turn conversation where each turn adds ~100 tokens of new content:
- Turn 1: ~100 input tokens
- Turn 50: ~5,000 input tokens
- **Total input tokens across all 50 turns: ~127,500** (vs. ~5,000 if you sent only the latest turn)

### What's Next?
The rest of this series addresses these limitations:
- **[02: Sliding Window Memory](../02_sliding_window_memory/)** keeps only the last *k* messages.
- **[03: Summary Memory](../03_summary_memory/)** replaces old turns with an LLM-generated summary.
- **[04: Summary Buffer Memory](../04_summary_buffer_memory/)** is a hybrid: summarize old messages, keep recent ones word-for-word.
- **[05: Token Buffer Memory](../05_token_buffer_memory/)** trims to a strict token budget.

## Further Reading

- [Anthropic Messages API: Multi-turn Conversations](https://docs.anthropic.com/en/docs/build-with-claude/conversational-ai?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)
- [LangChain ConversationBufferMemory](https://python.langchain.com/docs/modules/memory/types/buffer?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)
- [LlamaIndex Chat Store](https://docs.llamaindex.ai/en/stable/module_guides/storing/chat_stores/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)
- Anthropic's 7 Layers of Agent Memory (2026)
- [Lilian Weng, "LLM Powered Autonomous Agents" (Memory section)](https://lilianweng.github.io/posts/2023-06-23-agent/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)

---

*Next up: 02: Sliding Window Memory ->*

Clean up temporary files created during the demo.

In [ ]:
for f in ["conversation.json", "token_growth.png"]:
 if os.path.exists(f):
 os.remove(f)

## 🧪 Try It Yourself

Three small challenges to deepen your understanding. Each should take 10-30 minutes.

### Challenge 1: Multi-persona buffer
Create two separate `ConversationBufferMemory` instances with different system prompts (e.g., a tutor and a travel planner). Run five turns on each and compare how the buffer contents diverge. Pay attention to how the system prompt affects token counts.

### Challenge 2: Measure the cost curve
Extend the token-growth demo to 20 turns. Record `input_tokens` per turn, compute the cumulative cost in dollars (use Anthropic pricing for Claude Sonnet), and print a summary table. Compare the total cost against a hypothetical system that sends only the last 5 messages.

### Challenge 3: Swap in a database backend
Replace the JSON-based `save_conversation()` / `load_conversation()` with a SQLite backend. Store each message as a row with columns for role, content, and timestamp. Verify round-trip persistence still works. This prepares you for the patterns in 21 Cross-Session Memory.


![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--01-conversation-buffer-memory--conversation-buffer-memory)
